# Simulation study numbers for the paper

Recomputes quantitative claims in `Turnout_Paper/main.tex` (Simulation Experiments) and `Turnout_Paper/A2-simulation-study.tex` from `../tmp/models/model_results.csv`.

Run after updating simulation results. Each block prints **computed** values alongside the **paper** wording where applicable.

In [1]:
import json
import pathlib
import re

import numpy as np
import pandas as pd

FOCAL_MODELS = ['1_bp', '2_ei', '3_gg', '4_pm', '5_fs']
MODEL_LABELS = {
    '1_bp': 'BP',
    '2_ei': 'EI',
    '3_gg': 'GG',
    '4_pm': 'PM',
    '5_fs': 'FS',
}
METRICS = ['kld', 'kld_1d', 'kld_2d', 'emd', 'emd_1d', 'emd_2d']


def get_results(df, name, config_map=None, data_config=None):
    out = df.loc[df['data_name'] == name].copy()
    if config_map and data_config is not None:
        for config_name, col_name in config_map.items():
            out[col_name] = out['data_id'].apply(lambda data_id: data_config[data_id][config_name])
    return out


def median_by_model(df, metric, models=FOCAL_MODELS):
    return df.groupby('model_name')[metric].median().reindex(models)


def pct_reduction(baseline, improved):
    return (baseline - improved) / baseline * 100


def print_claim(label, computed, paper=None, fmt='.1f', suffix=''):
    text = f'{computed:{fmt}}{suffix}'
    if paper is None:
        print(f'  {label}: {text}')
    else:
        print(f'  {label}: {text}  (paper: {paper})')


def print_model_medians(df, metric, title=None):
    if title:
        print(title)
    med = median_by_model(df, metric)
    for model in FOCAL_MODELS:
        print(f'    {MODEL_LABELS[model]:>2} {metric}: {med[model]:.4f}')
    order = med.sort_values().index.tolist()
    print(f'    ordering (best→worst): {" > ".join(MODEL_LABELS[m] for m in order)}')
    return med


def selection_rate_for_data_id(data_id, data_dir=pathlib.Path('../tmp/data')):
    pop = pd.read_parquet(data_dir / data_id / 'population.parquet')
    if 'N' in pop.columns:
        return float((pop['selection'] * pop['N']).sum() / pop['N'].sum())
    return float(pop['selection'].mean())


def rates_by_intercept(non_response_df):
    rows = []
    for data_id in non_response_df['data_id'].unique():
        intercept = non_response_df.loc[non_response_df['data_id'] == data_id, 'selection_intercept'].iloc[0]
        rows.append({'data_id': data_id, 'selection_intercept': intercept, 'selection_rate': selection_rate_for_data_id(data_id)})
    rate_df = pd.DataFrame(rows)
    return rate_df.groupby('selection_intercept')['selection_rate'].median()

In [2]:
data_list = json.load(open('../tmp/data/data_list.json', 'r'))
data_config = {name: config for configs in data_list.values() for name, config in configs}

model_results_df = pd.read_csv('../tmp/models/model_results.csv', index_col=0)
print(f'Loaded {len(model_results_df)} rows from model_results.csv')
print(f'Scenarios: {sorted(model_results_df.data_name.unique())}')

Loaded 3872 rows from model_results.csv
Scenarios: ['est-agg-bias', 'est-default', 'est-electoral-district', 'est-hcoef-cor', 'est-hcoef-sigma', 'est-heck-cor', 'est-int', 'est-no-selection', 'est-noise-out', 'est-noise-sel', 'est-non-normal-error', 'est-non-response', 'est-overreport-const', 'est-sample-size']


## A2 — MCMC diagnostics and runtime

In [3]:
print('A2 / Model fitting')
print_claim('Grid rows in CSV', len(model_results_df), fmt='d', suffix='')
print_claim('Runs with divergences > 25', (model_results_df.divergences > 25).sum(), paper='0', fmt='d', suffix='')
print_claim('Max divergences', model_results_df.divergences.max(), paper='≤ 25', fmt='d', suffix='')
print_claim('Runs with mean R-hat > 1.1', (model_results_df.mean_rhat > 1.1).sum(), fmt='d', suffix='')
print_claim('Runs with mean R-hat > 1.2', (model_results_df.mean_rhat > 1.2).sum(), paper='numerous', fmt='d', suffix='')
if 'target_accept' in model_results_df.columns:
    print('  target_accept counts:', model_results_df.target_accept.value_counts().to_dict())
hours = model_results_df.fit_time.sum() / 3600
print_claim('Total fit_time (hours)', hours, paper='~320', fmt='.0f', suffix=' h')

A2 / Model fitting
  Grid rows in CSV: 3872
  Runs with divergences > 25: 0  (paper: 0)
  Max divergences: 25  (paper: ≤ 25)
  Runs with mean R-hat > 1.1: 205
  Runs with mean R-hat > 1.2: 129  (paper: numerous)
  target_accept counts: {0.9: 3609, 0.99: 263}
  Total fit_time (hours): 317 h  (paper: ~320)


## A2 — Realized data-generating rates

In [4]:
default_df = get_results(model_results_df, 'est-default')

print('A2 / Baseline DGP (est-default)')
print_claim('Realized selection rate (median)', default_df.selection_prop.median() * 100, paper='~21%', suffix='%')
print_claim('Realized population turnout (median topline)', default_df.pop_topline_yes.median() * 100, paper='~50%', suffix='%')
print_claim('Realized survey overreporting (median)', default_df.overreport_prop.median() * 100, paper='β_or=0.3 in DGP', suffix='%')

non_response_df = get_results(
    model_results_df, 'est-non-response',
    {'heckman_coef_kwargs/heckman_coefs/selection/intercept': 'selection_intercept'},
    data_config,
)
intercept_rates = rates_by_intercept(non_response_df)

print('\nA2 / Varying selection intercept β₀ˢ (median realized selection rate)')
paper_map = {-3.0: '~0.6%', -1.0: '~21%', 0.0: '~50%'}
for intercept in sorted(intercept_rates.index):
    paper = paper_map.get(intercept)
    print_claim(f'β₀ˢ={intercept:>4}', intercept_rates[intercept] * 100, paper=paper, suffix='%')

overreport_df = get_results(
    model_results_df, 'est-overreport-const',
    {'selection_kwargs/bias/offset': 'bias_offset'},
    data_config,
)
misreport = overreport_df.groupby('bias_offset').overreport_prop.median().sort_index()

print('\nA2 / Over-reporting grid (median realized misreport rate)')
paper_mis = {0.0: '0%', 0.3: '7%', 0.6: '14%', 0.9: '19%'}
for offset, rate in misreport.items():
    print_claim(f'β_OB={offset}', rate * 100, paper=paper_mis.get(offset), suffix='%')

A2 / Baseline DGP (est-default)
  Realized selection rate (median): 20.6%  (paper: ~21%)
  Realized population turnout (median topline): 48.7%  (paper: ~50%)
  Realized survey overreporting (median): 7.5%  (paper: β_or=0.3 in DGP)

A2 / Varying selection intercept β₀ˢ (median realized selection rate)
  β₀ˢ=-3.0: 0.6%  (paper: ~0.6%)
  β₀ˢ=-2.0: 5.0%
  β₀ˢ=-1.0: 20.6%  (paper: ~21%)
  β₀ˢ= 0.0: 48.9%  (paper: ~50%)
  β₀ˢ= 1.0: 79.4%
  β₀ˢ= 2.0: 95.1%
  β₀ˢ= 3.0: 99.4%

A2 / Over-reporting grid (median realized misreport rate)
  β_OB=0.0: 0.0%  (paper: 0%)
  β_OB=0.3: 7.5%  (paper: 7%)
  β_OB=0.6: 13.9%  (paper: 14%)
  β_OB=0.9: 18.8%  (paper: 19%)


## Main text — Baseline (`est-default`)

In [5]:
print('Main text / Baseline performance (median across 11 seeds)')
for metric in ['kld', 'emd']:
    med = print_model_medians(default_df, metric, title=f'\n  {metric.upper()}')

gg_kld = median_by_model(default_df, 'kld')['3_gg']
fs_kld = median_by_model(default_df, 'kld')['5_fs']
gg_emd = median_by_model(default_df, 'emd')['3_gg']
fs_emd = median_by_model(default_df, 'emd')['5_fs']

print('\n  Headline reductions (FS vs GG, full margin)')
print_claim('KLD reduction', pct_reduction(gg_kld, fs_kld), paper='57%', suffix='%')
print_claim('EMD reduction', pct_reduction(gg_emd, fs_emd), paper='36%', suffix='%')

Main text / Baseline performance (median across 11 seeds)

  KLD
    BP kld: 0.3275
    EI kld: 0.1700
    GG kld: 0.0180
    PM kld: 0.0118
    FS kld: 0.0078
    ordering (best→worst): FS > PM > GG > EI > BP

  EMD
    BP emd: 0.2778
    EI emd: 0.2307
    GG emd: 0.0601
    PM emd: 0.0500
    FS emd: 0.0383
    ordering (best→worst): FS > PM > GG > EI > BP

  Headline reductions (FS vs GG, full margin)
  KLD reduction: 56.5%  (paper: 57%)
  EMD reduction: 36.3%  (paper: 36%)


## Main text — Paired per-seed comparisons (`est-default`)

All models share the same 11 simulated datasets (seeds), so model comparisons can be
made *paired* (per seed) rather than by comparing unpaired medians. Paired analysis is
more powerful and is robust to seed-to-seed difficulty: it reports, for each model pair,
on how many seeds one model beats the other, the per-seed metric ratio, and a Wilcoxon
signed-rank test. A nonparametric bootstrap over seeds gives a Monte Carlo SE / CI for
the headline median reduction (Morris et al. 2019 recommend reporting Monte Carlo
uncertainty for simulation studies).

In [6]:
from scipy import stats


def paired_table(df, metric, pairs, n_boot=2000, seed=0):
    piv = df.pivot_table(index='data_id', columns='model_name', values=metric)
    rng = np.random.default_rng(seed)
    ids = piv.index.to_numpy()
    rows = []
    for a, b in pairs:
        p = piv[[a, b]].dropna()
        ratio = p[a] / p[b]
        wins = int((p[a] < p[b]).sum())
        wpval = stats.wilcoxon(p[a], p[b]).pvalue
        rows.append({
            'metric': metric,
            'comparison': f'{MODEL_LABELS[a]} vs {MODEL_LABELS[b]}',
            'better_wins': f'{MODEL_LABELS[a]} {wins}/{len(p)}',
            'ratio_median': round(ratio.median(), 3),
            'ratio_min': round(ratio.min(), 3),
            'ratio_max': round(ratio.max(), 3),
            'wilcoxon_p': round(wpval, 4),
        })
    return pd.DataFrame(rows)


PAIRS = [('5_fs', '3_gg'), ('5_fs', '4_pm'), ('4_pm', '3_gg')]
print('Paired per-seed comparisons on est-default (lower metric = better):\n')
for metric in ['kld', 'emd', 'kld_2d']:
    print(paired_table(default_df, metric, PAIRS).to_string(index=False))
    print()

# Bootstrap Monte Carlo CI for the headline FS-vs-GG median KLD reduction
piv = default_df.pivot_table(index='data_id', columns='model_name', values='kld')
ids = piv.index.to_numpy()
rng = np.random.default_rng(0)
point = (piv['3_gg'].median() - piv['5_fs'].median()) / piv['3_gg'].median()
boots = []
for _ in range(4000):
    s = piv.loc[rng.choice(ids, size=len(ids), replace=True)]
    boots.append((s['3_gg'].median() - s['5_fs'].median()) / s['3_gg'].median())
boots = np.array(boots)
print('Headline FS-vs-GG median KLD reduction (ratio of medians):')
print_claim('  point estimate', point * 100, paper='57%', suffix='%')
print_claim('  bootstrap SE', boots.std() * 100, suffix='%')
print(f'  bootstrap 95% CI: [{np.percentile(boots, 2.5) * 100:.0f}%, {np.percentile(boots, 97.5) * 100:.0f}%]')
print(f'  (paired median per-seed FS/GG KLD ratio: {(piv["5_fs"] / piv["3_gg"]).median():.3f})')


Paired per-seed comparisons on est-default (lower metric = better):

metric comparison better_wins  ratio_median  ratio_min  ratio_max  wilcoxon_p
   kld   FS vs GG    FS 10/11         0.302      0.143      1.122      0.0029
   kld   FS vs PM     FS 8/11         0.434      0.270      1.755      0.0537
   kld   PM vs GG    PM 11/11         0.696      0.434      0.810      0.0010

metric comparison better_wins  ratio_median  ratio_min  ratio_max  wilcoxon_p
   emd   FS vs GG     FS 9/11         0.537      0.390      1.176      0.0098
   emd   FS vs PM     FS 8/11         0.648      0.487      1.389      0.0537
   emd   PM vs GG    PM 11/11         0.856      0.668      0.938      0.0010

metric comparison better_wins  ratio_median  ratio_min  ratio_max  wilcoxon_p
kld_2d   FS vs GG    FS 10/11         0.285      0.139      1.071       0.002
kld_2d   FS vs PM     FS 8/11         0.400      0.200      1.759       0.042
kld_2d   PM vs GG    PM 11/11         0.674      0.417      0.808      

## Main text — No selection (`est-no-selection`)

In [7]:
no_sel_df = get_results(model_results_df, 'est-no-selection')
print('Main text / No selection bias')
bp_kld = median_by_model(no_sel_df, 'kld')['1_bp']
ei_kld = median_by_model(no_sel_df, 'kld')['2_ei']
print_claim('BP median KLD', bp_kld, suffix='')
print_claim('EI median KLD', ei_kld, suffix='')
print_claim('BP < EI?', bp_kld < ei_kld, paper='yes', suffix='')

Main text / No selection bias
  BP median KLD: 0.0
  EI median KLD: 0.2
  BP < EI?: 1.0  (paper: yes)


## Main text — Selection intercept / censoring (`est-non-response`)

In [8]:
base_2d = median_by_model(default_df, 'kld_2d')
base_kld = median_by_model(default_df, 'kld')

print('Main text / Selection intercept scenario')
for intercept, paper_rate in [(-3.0, '<1% accessibility'), (-1.0, '~21% accessibility')]:
    sub = non_response_df[non_response_df.selection_intercept == intercept]
    med_2d = median_by_model(sub, 'kld_2d')
    med_kld = median_by_model(sub, 'kld')
    sel_rate = sub.selection_prop.median() * 100
    print(f'\n  β₀ˢ={intercept} (median realized selection rate {sel_rate:.2f}%; paper: {paper_rate})')
    for model in ['3_gg', '4_pm', '5_fs']:
        print(f'    {MODEL_LABELS[model]} KLD_2d: {med_2d[model]:.4f}  (baseline {base_2d[model]:.4f})')
        print(f'    {MODEL_LABELS[model]} KLD full: {med_kld[model]:.4f}  (baseline {base_kld[model]:.4f})')
    if intercept == -3.0:
        pm_ratio = med_2d['4_pm'] / base_2d['4_pm']
        fs_ratio = med_2d['5_fs'] / base_2d['5_fs']
        print(f'    PM 2D vs baseline ratio: {pm_ratio:.2f}x  (paper: ~1x)')
        print(f'    FS 2D vs baseline ratio: {fs_ratio:.2f}x')
        print(f'    PM≈FS on 2D? {np.isclose(med_2d["4_pm"], med_2d["5_fs"], rtol=0.15)}  (paper: effectively identical)')
    if intercept == -1.0:
        print(f'    FS vs PM full KLD ratio: {med_kld["5_fs"] / med_kld["4_pm"]:.2f}  (paper: FS better)')
        print(f'    FS vs PM 2D KLD ratio: {med_2d["5_fs"] / med_2d["4_pm"]:.2f}')

Main text / Selection intercept scenario

  β₀ˢ=-3.0 (median realized selection rate 0.59%; paper: <1% accessibility)
    GG KLD_2d: 0.0091  (baseline 0.0049)
    GG KLD full: 0.0273  (baseline 0.0180)
    PM KLD_2d: 0.0030  (baseline 0.0031)
    PM KLD full: 0.0122  (baseline 0.0118)
    FS KLD_2d: 0.0030  (baseline 0.0018)
    FS KLD full: 0.0091  (baseline 0.0078)
    PM 2D vs baseline ratio: 0.99x  (paper: ~1x)
    FS 2D vs baseline ratio: 1.66x
    PM≈FS on 2D? True  (paper: effectively identical)

  β₀ˢ=-1.0 (median realized selection rate 20.58%; paper: ~21% accessibility)
    GG KLD_2d: 0.0055  (baseline 0.0049)
    GG KLD full: 0.0185  (baseline 0.0180)
    PM KLD_2d: 0.0032  (baseline 0.0031)
    PM KLD full: 0.0121  (baseline 0.0118)
    FS KLD_2d: 0.0018  (baseline 0.0018)
    FS KLD full: 0.0082  (baseline 0.0078)
    FS vs PM full KLD ratio: 0.68  (paper: FS better)
    FS vs PM 2D KLD ratio: 0.58


## Main text — Heckman error correlation (`est-heck-cor`)

In [9]:
heck_cor_df = get_results(
    model_results_df, 'est-heck-cor',
    {'heckman_coef_kwargs/heckman_coefs/error/rho': 'heckman_rho'},
    data_config,
)

print('Main text / Error correlation ρ')
for rho in sorted(heck_cor_df.heckman_rho.unique()):
    sub = heck_cor_df[heck_cor_df.heckman_rho == rho]
    med = median_by_model(sub, 'kld')
    print(f'\n  ρ={rho}')
    for model in ['3_gg', '4_pm', '5_fs']:
        print(f'    {MODEL_LABELS[model]} KLD: {med[model]:.4f}')

rho0 = median_by_model(heck_cor_df[heck_cor_df.heckman_rho == 0.0], 'kld')
rho_hi = heck_cor_df[heck_cor_df.heckman_rho >= 0.75].groupby('model_name')['kld'].median().reindex(FOCAL_MODELS)
print('\n  ρ=0 spread (GG−FS):', rho0['3_gg'] - rho0['5_fs'])
print('  High ρ spread (GG−FS):', rho_hi['3_gg'] - rho_hi['5_fs'], '  (paper: GG degrades more)')
print('  FS lowest at high ρ?', rho_hi['5_fs'] == rho_hi[['3_gg', '4_pm', '5_fs']].min(), '  (paper: yes)')

Main text / Error correlation ρ

  ρ=0.0
    GG KLD: 0.0073
    PM KLD: 0.0067
    FS KLD: 0.0069

  ρ=0.25
    GG KLD: 0.0091
    PM KLD: 0.0056
    FS KLD: 0.0065

  ρ=0.5
    GG KLD: 0.0182
    PM KLD: 0.0123
    FS KLD: 0.0076

  ρ=0.75
    GG KLD: 0.0411
    PM KLD: 0.0274
    FS KLD: 0.0063

  ρ=1.0
    GG KLD: 0.6494
    PM KLD: 0.1038
    FS KLD: 0.0106

  ρ=0 spread (GG−FS): 0.0004570489028398999
  High ρ spread (GG−FS): 0.09672532197647644   (paper: GG degrades more)
  FS lowest at high ρ? True   (paper: yes)


## Main text — Selection noise (`est-noise-sel`)

In [10]:
noise_sel_df = get_results(
    model_results_df, 'est-noise-sel',
    {'heckman_coef_kwargs/heckman_coefs/selection_noise_prop': 'selection_noise_prop'},
    data_config,
)

print('Main text / Random selection noise')
for noise in sorted(noise_sel_df.selection_noise_prop.unique()):
    sub = noise_sel_df[noise_sel_df.selection_noise_prop == noise]
    med = median_by_model(sub, 'kld')
    print(f'\n  selection_noise_prop={noise}')
    for model in ['3_gg', '4_pm', '5_fs']:
        print(f'    {MODEL_LABELS[model]} KLD: {med[model]:.4f}')

noise0 = median_by_model(noise_sel_df[noise_sel_df.selection_noise_prop == 0.0], 'kld')
noise10 = median_by_model(noise_sel_df[noise_sel_df.selection_noise_prop == 0.10], 'kld')
fs_edge_0 = noise0['3_gg'] - noise0['5_fs']
fs_edge_10 = noise10['3_gg'] - noise10['5_fs']
print('\n  FS edge over GG at 0% noise:', fs_edge_0)
print('  FS edge over GG at 10% noise:', fs_edge_10, '  (paper: edge disappears)')
print('  FS≈PM at 10%?', np.isclose(noise10['5_fs'], noise10['4_pm'], rtol=0.1), f"(FS={noise10['5_fs']:.4f}, PM={noise10['4_pm']:.4f})")

Main text / Random selection noise

  selection_noise_prop=0.0
    GG KLD: 0.0177
    PM KLD: 0.0122
    FS KLD: 0.0075

  selection_noise_prop=0.05
    GG KLD: 0.0121
    PM KLD: 0.0083
    FS KLD: 0.0084

  selection_noise_prop=0.1
    GG KLD: 0.0089
    PM KLD: 0.0066
    FS KLD: 0.0066

  selection_noise_prop=0.2
    GG KLD: 0.0084
    PM KLD: 0.0063
    FS KLD: 0.0072

  FS edge over GG at 0% noise: 0.0101948171855571
  FS edge over GG at 10% noise: 0.002325719090509301   (paper: edge disappears)
  FS≈PM at 10%? True (FS=0.0066, PM=0.0066)


## Main text — Sample size (`est-sample-size`)

In [11]:
sample_df = get_results(
    model_results_df, 'est-sample-size',
    {'selection_kwargs/sample_size': 'sample_size'},
    data_config,
)

print('Main text / Sample size (median KLD)')
for n in sorted(sample_df.sample_size.unique()):
    sub = sample_df[sample_df.sample_size == n]
    med = median_by_model(sub, 'kld')
    print(f'  N_S={int(n):>4}: GG {med["3_gg"]:.4f}  PM {med["4_pm"]:.4f}  FS {med["5_fs"]:.4f}')

small = sample_df[sample_df.sample_size == sample_df.sample_size.min()]
large = sample_df[sample_df.sample_size == sample_df.sample_size.max()]
small_med = median_by_model(small, 'kld')
large_med = median_by_model(large, 'kld')
n_small, n_large = int(sample_df.sample_size.min()), int(sample_df.sample_size.max())
print(f'\n  Proportional KLD reduction N_S={n_small}→{n_large} (matches log-scale figure; not absolute drop):')
for model in ['3_gg', '4_pm', '5_fs']:
    rel = pct_reduction(small_med[model], large_med[model])
    factor = small_med[model] / large_med[model]
    print(f'    {MODEL_LABELS[model]}: {rel:.1f}% ({factor:.1f}× lower)')
best = max(['3_gg', '4_pm', '5_fs'], key=lambda m: pct_reduction(small_med[m], large_med[m]))
print(f'  steepest proportional decline: {MODEL_LABELS[best]}  (paper: FS benefits most from more data)')
print('  note: absolute KLD drop ranks GG > PM > FS here')

Main text / Sample size (median KLD)
  N_S= 100: GG 0.0549  PM 0.0387  FS 0.0310
  N_S= 250: GG 0.0217  PM 0.0178  FS 0.0163
  N_S= 500: GG 0.0178  PM 0.0122  FS 0.0108
  N_S=1000: GG 0.0179  PM 0.0122  FS 0.0074
  N_S=2000: GG 0.0173  PM 0.0097  FS 0.0043

  Proportional KLD reduction N_S=100→2000 (matches log-scale figure; not absolute drop):
    GG: 68.5% (3.2× lower)
    PM: 75.0% (4.0× lower)
    FS: 86.0% (7.1× lower)
  steepest proportional decline: FS  (paper: FS benefits most from more data)
  note: absolute KLD drop ranks GG > PM > FS here


## Main text — Margin informativeness (`est-electoral-district`)

In [12]:
margin_name_map = {
    '2_margin_tpl_ei': '2_ei', '3_margin_tpl_gg': '3_gg', '4_margin_tpl_pm': '4_pm', '5_margin_tpl_fs': '5_fs',
    '2_margin_electoral_district_ei': '2_ei', '3_margin_electoral_district_gg': '3_gg',
    '4_margin_electoral_district_pm': '4_pm', '5_margin_electoral_district_fs': '5_fs',
    '2_margin_municipality_ei': '2_ei', '3_margin_municipality_gg': '3_gg',
    '4_margin_municipality_pm': '4_pm', '5_margin_municipality_fs': '5_fs',
}
margin_level_map = {
    '2_margin_tpl_ei': 'topline', '3_margin_tpl_gg': 'topline', '4_margin_tpl_pm': 'topline', '5_margin_tpl_fs': 'topline',
    '2_margin_electoral_district_ei': 'electoral_district', '3_margin_electoral_district_gg': 'electoral_district',
    '4_margin_electoral_district_pm': 'electoral_district', '5_margin_electoral_district_fs': 'electoral_district',
    '2_margin_municipality_ei': 'municipality', '3_margin_municipality_gg': 'municipality',
    '4_margin_municipality_pm': 'municipality', '5_margin_municipality_fs': 'municipality',
}

margin_df = get_results(model_results_df, 'est-electoral-district')
margin_df = margin_df[margin_df.model_name.isin(margin_name_map)].copy()
margin_df['base_model'] = margin_df.model_name.map(margin_name_map)
margin_df['margin_level'] = margin_df.model_name.map(margin_level_map)

print('Main text / Margin informativeness (median KLD)')
for level in ['topline', 'electoral_district', 'municipality']:
    sub = margin_df[margin_df.margin_level == level]
    med = sub.groupby('base_model')['kld'].median().reindex(['3_gg', '4_pm', '5_fs'])
    print(f'  {level:>20}: GG {med["3_gg"]:.4f}  PM {med["4_pm"]:.4f}  FS {med["5_fs"]:.4f}')

tpl = margin_df[margin_df.margin_level == 'topline'].groupby('base_model')['kld'].median().reindex(['3_gg', '4_pm', '5_fs'])
mun = margin_df[margin_df.margin_level == 'municipality'].groupby('base_model')['kld'].median().reindex(['3_gg', '4_pm', '5_fs'])
print('\n  Proportional KLD reduction topline→municipality (matches log-scale figure; not absolute drop):')
for model in ['3_gg', '4_pm', '5_fs']:
    rel = pct_reduction(tpl[model], mun[model])
    factor = tpl[model] / mun[model]
    print(f'    {MODEL_LABELS[model]}: {rel:.1f}% ({factor:.1f}× lower)')
best = max(['3_gg', '4_pm', '5_fs'], key=lambda m: pct_reduction(tpl[m], mun[m]))
print(f'  steepest proportional decline: {MODEL_LABELS[best]}  (paper: FS largest proportional gains)')
print('  note: absolute KLD drop ranks PM > FS > GG here')

Main text / Margin informativeness (median KLD)
               topline: GG 0.0416  PM 0.0327  FS 0.0173
    electoral_district: GG 0.0357  PM 0.0233  FS 0.0108
          municipality: GG 0.0325  PM 0.0194  FS 0.0072

  Proportional KLD reduction topline→municipality (matches log-scale figure; not absolute drop):
    GG: 21.9% (1.3× lower)
    PM: 40.7% (1.7× lower)
    FS: 58.4% (2.4× lower)
  steepest proportional decline: FS  (paper: FS largest proportional gains)
  note: absolute KLD drop ranks PM > FS > GG here


## Sanity checks — stress scenarios described as unproblematic

In [13]:
print('Main text / Robustness scenarios (median full KLD, focal models)')
for scenario, label in [
    ('est-hcoef-cor', 'collinearity'),
    ('est-agg-bias', 'aggregation bias'),
    ('est-non-normal-error', 'skewed-t errors'),
    ('est-int', 'interaction effects'),
]:
    sub = get_results(model_results_df, scenario)
    med = median_by_model(sub, 'kld')
    print(f'  {label}: ' + ', '.join(f'{MODEL_LABELS[m]} {med[m]:.3f}' for m in FOCAL_MODELS))

Main text / Robustness scenarios (median full KLD, focal models)
  collinearity: BP 0.356, EI 0.157, GG 0.020, PM 0.013, FS 0.007
  aggregation bias: BP 0.291, EI 0.171, GG 0.018, PM 0.014, FS 0.008
  skewed-t errors: BP 0.173, EI 0.086, GG 0.016, PM 0.012, FS 0.010
  interaction effects: BP 0.235, EI 0.222, GG 0.150, PM 0.149, FS 0.145
